# 🌾 Ferme Agricole — Analyse Machine Learning
## Amélioration de la Production par l'Enrichissement des Données

**Notebook académique — Big Data & Agriculture de Précision**

---
### Plan du Notebook
1. Imports & Configuration
2. Chargement et aperçu des données enrichies
3. Analyse Exploratoire (EDA)
4. Préparation des données (Feature Engineering)
5. Modèle 1 — Random Forest : Prédiction du rendement
6. Modèle 2 — K-Means : Segmentation des parcelles
7. Modèle 3 — Prophet : Prévision des prix de marché
8. Modèle 4 — Isolation Forest : Détection d'anomalies
9. Synthèse et recommandations


## 1. Imports & Configuration

In [ ]:
# Manipulation des données
import pandas as pd
import numpy as np
import os, warnings
warnings.filterwarnings('ignore')

# Visualisation
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.size'] = 11
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
PALETTE = ['#1F5C2E','#2E8B57','#7FB069','#F4A261','#E76F51']

# Machine Learning
from sklearn.ensemble import RandomForestRegressor, IsolationForest
from sklearn.cluster import KMeans
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.metrics import (mean_absolute_error, r2_score,
                             mean_squared_error, silhouette_score)
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.pipeline import Pipeline
import joblib

# Chemins
DATA_ENRICHED = '../data/enriched'
MODELS_DIR    = '../data/models'
os.makedirs(MODELS_DIR, exist_ok=True)

print("✓ Tous les imports chargés")


## 2. Chargement et Aperçu des Données Enrichies

In [ ]:
df = pd.read_csv(os.path.join(DATA_ENRICHED, 'dataset_enrichi.csv'),
                  parse_dates=['date_semis','date_recolte'])

print(f"Shape : {df.shape}")
print(f"Colonnes : {list(df.columns)}")
df.head()


In [ ]:
# Types et valeurs manquantes
print("── Types de données ──")
print(df.dtypes)
print(f"\n── Valeurs manquantes ──")
missing = df.isnull().sum()
print(missing[missing > 0] if missing.sum() > 0 else "Aucune valeur manquante ✓")


In [ ]:
# Statistiques descriptives
df.describe().round(3)


## 3. Analyse Exploratoire (EDA)

L'EDA est une étape **incontournable** en Data Science. Elle permet de :
- Comprendre la distribution des variables
- Détecter les outliers et corrélations
- Formuler des hypothèses avant la modélisation


In [ ]:
# Distribution des rendements par culture
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Boxplot
df.boxplot(column='rendement_t_ha', by='culture', ax=axes[0],
           boxprops=dict(color='#2E8B57'), medianprops=dict(color='#E76F51', linewidth=2))
axes[0].set_title('Distribution des rendements par culture')
axes[0].set_xlabel('Culture'); axes[0].set_ylabel('Rendement (t/ha)')
plt.sca(axes[0]); plt.xticks(rotation=30)

# Rendement moyen
rend_moy = df.groupby('culture')['rendement_t_ha'].mean().sort_values(ascending=True)
rend_moy.plot(kind='barh', ax=axes[1], color=PALETTE)
axes[1].set_title('Rendement moyen par culture')
axes[1].set_xlabel('t/ha')

plt.suptitle('')
plt.tight_layout()
plt.savefig(os.path.join(MODELS_DIR, 'eda_rendements.png'), dpi=150, bbox_inches='tight')
plt.show()
print(rend_moy.round(2).to_string())


In [ ]:
# Matrice de corrélation — variables numériques clés
cols_corr = [
    'rendement_t_ha', 'indice_fertilite_sol', 'score_risque_climatique',
    'meteo_pluie_totale_mm', 'meteo_bilan_hydrique', 'meteo_jours_stress',
    'engrais_kg_ha', 'ph_sol', 'azote_N_ppm', 'matiere_organique_pct',
    'marge_nette_fcfa_ha'
]

corr = df[cols_corr].corr()

fig, ax = plt.subplots(figsize=(12, 9))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', center=0,
            cmap='RdYlGn', ax=ax, linewidths=0.5,
            annot_kws={'size': 9})
ax.set_title('Matrice de corrélation — Variables enrichies', fontsize=14, pad=20)
plt.tight_layout()
plt.savefig(os.path.join(MODELS_DIR, 'eda_correlation.png'), dpi=150, bbox_inches='tight')
plt.show()

# Top corrélations avec le rendement
print("── Top corrélations avec rendement_t_ha ──")
print(corr['rendement_t_ha'].drop('rendement_t_ha').abs()
      .sort_values(ascending=False).head(8).round(3))


In [ ]:
# Analyse marge nette vs rendement
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scatter rendement vs marge
for i, (cult, grp) in enumerate(df.groupby('culture')):
    axes[0].scatter(grp['rendement_t_ha'], grp['marge_nette_fcfa_ha']/1000,
                    label=cult, alpha=0.7, s=60, color=PALETTE[i % len(PALETTE)])
axes[0].set_xlabel('Rendement (t/ha)')
axes[0].set_ylabel('Marge nette (FCFA k/ha)')
axes[0].set_title('Rendement vs Marge nette')
axes[0].legend(fontsize=9)

# Marge par saison
marge_saison = df.groupby(['saison','annee'])['marge_nette_fcfa_ha'].mean().reset_index()
marge_saison['label'] = marge_saison['annee'].astype(str) + ' — ' + marge_saison['saison']
axes[1].bar(marge_saison['label'], marge_saison['marge_nette_fcfa_ha']/1000,
            color=PALETTE)
axes[1].set_title('Marge nette moyenne par saison')
axes[1].set_ylabel('FCFA (×1000)/ha')
plt.xticks(rotation=30)

plt.tight_layout()
plt.show()


In [ ]:
# Alertes — vue d'ensemble
nb_h = df['alerte_deficit_hydrique'].sum()
nb_t = df['alerte_stress_thermique'].sum()
total = len(df)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Pie alertes
labels = [f'Déficit Hydrique\n({nb_h})', f'Stress Thermique\n({nb_t})',
          f'Aucune alerte\n({total - nb_h - nb_t})']
sizes  = [nb_h, nb_t, total - nb_h - nb_t]
colors = ['#dc2626','#ea580c','#16a34a']
axes[0].pie(sizes, labels=labels, colors=colors, autopct='%1.0f%%',
            startangle=90, textprops={'fontsize':10})
axes[0].set_title('Répartition des alertes')

# Alertes par parcelle
alertes_parc = df.groupby('parcelle_id').agg(
    hydrique=('alerte_deficit_hydrique','sum'),
    thermique=('alerte_stress_thermique','sum')
).reset_index()
x = range(len(alertes_parc))
axes[1].bar(x, alertes_parc['hydrique'],  label='Déficit hydrique',  color='#dc2626', alpha=0.8)
axes[1].bar(x, alertes_parc['thermique'], label='Stress thermique',  color='#ea580c', alpha=0.8,
            bottom=alertes_parc['hydrique'])
axes[1].set_xticks(x); axes[1].set_xticklabels(alertes_parc['parcelle_id'], rotation=30)
axes[1].set_title('Alertes par parcelle'); axes[1].legend()

plt.tight_layout()
plt.savefig(os.path.join(MODELS_DIR, 'eda_alertes.png'), dpi=150, bbox_inches='tight')
plt.show()


## 4. Préparation des Données (Feature Engineering)

Étapes :
1. Sélection des features pertinentes (basé sur l'EDA et la connaissance métier)
2. Encodage des variables catégorielles
3. Gestion des valeurs manquantes
4. Split train/test stratifié


In [ ]:
# Sélection des features
FEATURES = [
    # Variables sol
    'indice_fertilite_sol', 'ph_sol', 'azote_N_ppm',
    'phosphore_P_ppm', 'potassium_K_ppm', 'matiere_organique_pct',
    # Variables météo
    'score_risque_climatique', 'meteo_temp_moy_cycle',
    'meteo_pluie_totale_mm', 'meteo_bilan_hydrique',
    'meteo_jours_stress', 'meteo_ensoleillement_moy_h',
    'duree_cycle_jours',
    # Variables intrants
    'engrais_kg_ha', 'eau_irrigation_m3_ha',
    # Variable cible encodée
    'culture_enc',
]
TARGET = 'rendement_t_ha'

# Encodage de la culture
le = LabelEncoder()
df_ml = df.copy()
df_ml['culture_enc'] = le.fit_transform(df_ml['culture'])

print(f"Classes encodées : {dict(zip(le.classes_, le.transform(le.classes_)))}")

# Dataset final
df_final = df_ml[FEATURES + [TARGET, 'culture', 'parcelle_id']].dropna()
print(f"\nDataset final : {df_final.shape}")
print(f"Distribution du target :")
print(df_final[TARGET].describe().round(3))


In [ ]:
# Split Train / Test (75% / 25%)
X = df_final[FEATURES]
y = df_final[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)

print(f"Train : {X_train.shape[0]} échantillons ({X_train.shape[0]/len(X)*100:.0f}%)")
print(f"Test  : {X_test.shape[0]} échantillons  ({X_test.shape[0]/len(X)*100:.0f}%)")
print(f"\nDistribution y_train : moy={y_train.mean():.2f}, std={y_train.std():.2f}")
print(f"Distribution y_test  : moy={y_test.mean():.2f}, std={y_test.std():.2f}")


## 5. Modèle 1 — Random Forest : Prédiction du Rendement

**Choix du modèle :**
Le Random Forest est particulièrement adapté à ce problème car :
- Il gère les relations non-linéaires (interactions sol × météo × intrants)
- Il est robuste aux outliers
- Il fournit l'importance des variables (interprétabilité)
- Il n'exige pas de normalisation des données


In [ ]:
# ── Entraînement du modèle de base ──
rf_base = RandomForestRegressor(
    n_estimators=200,
    max_depth=8,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1
)
rf_base.fit(X_train, y_train)

y_pred_base = rf_base.predict(X_test)

mae_base  = mean_absolute_error(y_test, y_pred_base)
rmse_base = np.sqrt(mean_squared_error(y_test, y_pred_base))
r2_base   = r2_score(y_test, y_pred_base)

print("── Métriques Modèle de Base ──")
print(f"MAE  : {mae_base:.3f} t/ha")
print(f"RMSE : {rmse_base:.3f} t/ha")
print(f"R²   : {r2_base:.3f}")


In [ ]:
# ── Validation croisée (5-fold) ──
cv_r2   = cross_val_score(rf_base, X, y, cv=5, scoring='r2')
cv_mae  = cross_val_score(rf_base, X, y, cv=5, scoring='neg_mean_absolute_error')

print("── Cross-Validation 5-fold ──")
print(f"R²  : {cv_r2.mean():.3f} ± {cv_r2.std():.3f}  (min={cv_r2.min():.3f}, max={cv_r2.max():.3f})")
print(f"MAE : {(-cv_mae).mean():.3f} ± {(-cv_mae).std():.3f}")

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(range(1,6), cv_r2, color=PALETTE, alpha=0.85)
ax.axhline(cv_r2.mean(), color='red', linestyle='--', label=f'Moy. R² = {cv_r2.mean():.3f}')
ax.set_xlabel('Fold'); ax.set_ylabel('R²')
ax.set_title('Validation croisée 5-fold — Random Forest')
ax.set_xticks(range(1,6))
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(MODELS_DIR, 'rf_cv_scores.png'), dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ── Importance des variables ──
importances = pd.DataFrame({
    'feature':    FEATURES,
    'importance': rf_base.feature_importances_
}).sort_values('importance', ascending=True)

fig, ax = plt.subplots(figsize=(10, 6))
colors = ['#E76F51' if imp > 0.1 else '#2E8B57' if imp > 0.05 else '#7FB069'
          for imp in importances['importance']]
ax.barh(importances['feature'], importances['importance'], color=colors)
ax.set_title('Importance des variables — Random Forest', fontsize=13)
ax.set_xlabel('Importance (Mean Decrease Impurity)')
ax.axvline(importances['importance'].mean(), color='red', linestyle='--', alpha=0.6,
           label=f'Moy. = {importances.importance.mean():.3f}')
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(MODELS_DIR, 'rf_importance.png'), dpi=150, bbox_inches='tight')
plt.show()

print("Top 5 variables les plus importantes :")
print(importances.tail(5)[['feature','importance']].iloc[::-1].to_string(index=False))


In [ ]:
# ── Graphique Réel vs Prédit ──
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scatter
axes[0].scatter(y_test, y_pred_base, alpha=0.7, color='#2E8B57', s=60, edgecolors='white')
lims = [min(y_test.min(), y_pred_base.min()), max(y_test.max(), y_pred_base.max())]
axes[0].plot(lims, lims, 'r--', lw=2, label='Prédiction parfaite')
axes[0].set_xlabel('Rendement réel (t/ha)')
axes[0].set_ylabel('Rendement prédit (t/ha)')
axes[0].set_title(f'Réel vs Prédit — R² = {r2_base:.3f}')
axes[0].legend()

# Résidus
residus = y_test.values - y_pred_base
axes[1].scatter(y_pred_base, residus, alpha=0.7, color='#F4A261', s=60, edgecolors='white')
axes[1].axhline(0, color='red', linestyle='--')
axes[1].set_xlabel('Valeurs prédites (t/ha)')
axes[1].set_ylabel('Résidus (t/ha)')
axes[1].set_title(f'Analyse des résidus — MAE = {mae_base:.3f}')

plt.tight_layout()
plt.savefig(os.path.join(MODELS_DIR, 'rf_predictions.png'), dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ── Sauvegarde du modèle ──
joblib.dump(rf_base, os.path.join(MODELS_DIR, 'rf_rendement.pkl'))
joblib.dump(le,      os.path.join(MODELS_DIR, 'label_encoder_culture.pkl'))
importances.to_csv(os.path.join(MODELS_DIR, 'rf_importances.csv'), index=False)

print("✓ Modèle Random Forest sauvegardé → rf_rendement.pkl")
print(f"  R² = {r2_base:.3f} | MAE = {mae_base:.3f} t/ha | RMSE = {rmse_base:.3f} t/ha")


## 6. Modèle 2 — K-Means : Segmentation des Parcelles

**Objectif :** Identifier des groupes de parcelles aux profils similaires afin d'adapter les stratégies agricoles par segment.

**Méthode du coude (Elbow Method) :** Déterminer le nombre optimal de clusters.


In [ ]:
# Agrégation par parcelle
agg = df_ml.groupby('parcelle_id').agg(
    rendement_moy=('rendement_t_ha','mean'),
    rendement_std=('rendement_t_ha','std'),
    marge_moy=('marge_nette_fcfa_ha','mean'),
    indice_sol=('indice_fertilite_sol','mean'),
    risque_moy=('score_risque_climatique','mean'),
    nb_alertes=('alerte_deficit_hydrique','sum'),
    culture=('culture','first'),
).reset_index().fillna(0)

features_seg = ['rendement_moy','rendement_std','marge_moy','indice_sol','risque_moy','nb_alertes']

scaler   = StandardScaler()
X_scaled = scaler.fit_transform(agg[features_seg])

# ── Méthode du coude ──
inertias = []
K_range  = range(2, 8)
for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_scaled)
    inertias.append(km.inertia_)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(K_range, inertias, 'o-', color='#2E8B57', linewidth=2, markersize=8)
axes[0].set_xlabel('Nombre de clusters (k)')
axes[0].set_ylabel('Inertie')
axes[0].set_title('Méthode du Coude — Choix de k')
axes[0].axvline(3, color='red', linestyle='--', label='k=3 optimal')
axes[0].legend()

# Score de silhouette
sil_scores = []
for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_scaled)
    sil_scores.append(silhouette_score(X_scaled, labels))

axes[1].plot(K_range, sil_scores, 's-', color='#F4A261', linewidth=2, markersize=8)
axes[1].set_xlabel('Nombre de clusters (k)')
axes[1].set_ylabel('Score Silhouette')
axes[1].set_title('Score Silhouette — Plus élevé = meilleur')
axes[1].axvline(3, color='red', linestyle='--', label=f'k=3 (s={sil_scores[1]:.3f})')
axes[1].legend()

plt.tight_layout()
plt.savefig(os.path.join(MODELS_DIR, 'kmeans_elbow.png'), dpi=150, bbox_inches='tight')
plt.show()
print(f"Score silhouette pour k=3 : {sil_scores[1]:.4f}")


In [ ]:
# ── Entraînement K-Means k=3 ──
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
agg['segment'] = kmeans.fit_predict(X_scaled)

# Réordonner segments par rendement décroissant
ordre   = agg.groupby('segment')['rendement_moy'].mean().sort_values(ascending=False)
mapping = {old: new for new, old in enumerate(ordre.index)}
agg['segment'] = agg['segment'].map(mapping)

labels_seg  = {0:'Haute Performance', 1:'Performance Moyenne', 2:'Sous-Performance'}
couleurs_seg = {0:'#16a34a', 1:'#ca8a04', 2:'#dc2626'}
agg['segment_label'] = agg['segment'].map(labels_seg)
agg['couleur']       = agg['segment'].map(couleurs_seg)

# Profil de chaque segment
print("── Profil des segments ──")
print(agg.groupby('segment_label')[features_seg].mean().round(2).to_string())


In [ ]:
# ── Visualisation des clusters ──
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scatter rendement vs marge
for seg, label in labels_seg.items():
    g = agg[agg['segment'] == seg]
    axes[0].scatter(g['rendement_moy'], g['marge_moy']/1000,
                    c=couleurs_seg[seg], label=label, s=120, alpha=0.85, edgecolors='white', zorder=3)
    for _, row in g.iterrows():
        axes[0].annotate(row['parcelle_id'], (row['rendement_moy'], row['marge_moy']/1000),
                         fontsize=8, ha='center', va='bottom')

axes[0].set_xlabel('Rendement moyen (t/ha)')
axes[0].set_ylabel('Marge moyenne (FCFA k/ha)')
axes[0].set_title('Segmentation K-Means des parcelles')
axes[0].legend()

# Radar-like : profil par segment
cats = ['rendement_moy','marge_moy','indice_sol','risque_moy']
cats_labels = ['Rendement','Marge','Sol','Risque']
for i, (seg, label) in enumerate(labels_seg.items()):
    vals = agg[agg['segment']==seg][cats].mean().values
    vals_norm = vals / agg[cats].max().values
    axes[1].bar(np.arange(len(cats)) + i*0.25, vals_norm, 0.25,
                label=label, color=couleurs_seg[seg], alpha=0.85)

axes[1].set_xticks(np.arange(len(cats)) + 0.25)
axes[1].set_xticklabels(cats_labels)
axes[1].set_title('Profil normalisé des segments')
axes[1].set_ylabel('Score normalisé')
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.savefig(os.path.join(MODELS_DIR, 'kmeans_clusters.png'), dpi=150, bbox_inches='tight')
plt.show()

# Sauvegarde
agg.to_csv(os.path.join(MODELS_DIR, 'segmentation_parcelles.csv'), index=False)
joblib.dump(kmeans, os.path.join(MODELS_DIR, 'kmeans_parcelles.pkl'))
print("✓ Segmentation sauvegardée → segmentation_parcelles.csv")


## 7. Modèle 3 — Prophet : Prévision des Prix de Marché

**Prophet** (Meta) est conçu pour les séries temporelles avec saisonnalité. Il est particulièrement adapté aux données agricoles car les prix suivent des cycles saisonniers.


In [ ]:
from prophet import Prophet
from prophet.plot import plot_plotly

df_marche = pd.read_csv(os.path.join(DATA_ENRICHED, 'marche.csv'), parse_dates=['semaine'])

# Démonstration sur le Maïs
culture_demo = 'Maïs'
df_mais = df_marche[df_marche['culture'] == culture_demo][['semaine','prix_fcfa_kg']].copy()
df_mais.columns = ['ds', 'y']
df_mais = df_mais.sort_values('ds')

print(f"Série temporelle {culture_demo} : {len(df_mais)} semaines")
print(df_mais.tail())


In [ ]:
# Entraînement Prophet
model_prophet = Prophet(
    yearly_seasonality=True,
    weekly_seasonality=False,
    daily_seasonality=False,
    seasonality_mode='multiplicative',
    changepoint_prior_scale=0.1,
    interval_width=0.90
)
model_prophet.fit(df_mais)

# Prévision 26 semaines
future   = model_prophet.make_future_dataframe(periods=26, freq='W')
forecast = model_prophet.predict(future)

# Visualisation
fig, axes = plt.subplots(2, 1, figsize=(13, 8))

# Série + prévision
axes[0].plot(df_mais['ds'], df_mais['y'], 'o', color='#2E8B57', ms=4, label='Données historiques')
axes[0].plot(forecast['ds'], forecast['yhat'], '-', color='#E76F51', lw=2, label='Prévision')
axes[0].fill_between(forecast['ds'], forecast['yhat_lower'], forecast['yhat_upper'],
                      alpha=0.2, color='#E76F51', label='IC 90%')
axes[0].axvline(df_mais['ds'].max(), color='gray', linestyle='--', alpha=0.6, label='Début prévision')
axes[0].set_title(f'Prévision des prix — {culture_demo} (26 semaines)')
axes[0].set_ylabel('FCFA/kg'); axes[0].legend(fontsize=9)

# Composantes
components = model_prophet.plot_components(forecast)
plt.savefig(os.path.join(MODELS_DIR, 'prophet_mais.png'), dpi=150, bbox_inches='tight')
plt.show()

# Métriques sur données connues
from sklearn.metrics import mean_absolute_percentage_error
y_known = forecast[forecast['ds'].isin(df_mais['ds'])]['yhat'].values
mape = mean_absolute_percentage_error(df_mais['y'], y_known)
print(f"MAPE (in-sample) : {mape*100:.2f}%")


In [ ]:
# Prévisions pour toutes les cultures
previsions_all = []
for culture in df_marche['culture'].unique():
    df_c = df_marche[df_marche['culture']==culture][['semaine','prix_fcfa_kg']].copy()
    df_c.columns = ['ds','y']
    m = Prophet(yearly_seasonality=True, seasonality_mode='multiplicative',
                weekly_seasonality=False, daily_seasonality=False)
    m.fit(df_c.sort_values('ds'))
    future = m.make_future_dataframe(periods=26, freq='W')
    fc = m.predict(future)
    fc['culture'] = culture
    previsions_all.append(fc[['ds','yhat','yhat_lower','yhat_upper','culture']].tail(26))

df_prev = pd.concat(previsions_all, ignore_index=True)
df_prev.columns = ['date','prix_predit','prix_min','prix_max','culture']
df_prev.to_csv(os.path.join(MODELS_DIR, 'previsions_prix.csv'), index=False)
print(f"✓ Prévisions sauvegardées pour {df_marche['culture'].nunique()} cultures")


## 8. Modèle 4 — Isolation Forest : Détection d'Anomalies

**Isolation Forest** détecte les anomalies en isolant les observations atypiques. Utile pour identifier des cycles de production anormaux (parasites, erreurs de saisie, conditions extrêmes).


In [ ]:
features_ano = ['rendement_t_ha','engrais_kg_ha','eau_irrigation_m3_ha',
                'meteo_pluie_totale_mm','meteo_jours_stress']

df_ano  = df[features_ano].dropna()
iso     = IsolationForest(contamination=0.05, random_state=42)
preds   = iso.fit_predict(df_ano)
scores  = iso.score_samples(df_ano)

df_result = df.loc[df_ano.index].copy()
df_result['anomalie']       = (preds == -1).astype(int)
df_result['score_anomalie'] = scores.round(4)

nb_ano = df_result['anomalie'].sum()
print(f"Anomalies détectées : {nb_ano} / {len(df_result)} ({nb_ano/len(df_result)*100:.1f}%)")
print(f"\nDétail des anomalies :")
print(df_result[df_result['anomalie']==1][['parcelle_id','culture','saison','annee',
                                            'rendement_t_ha','score_anomalie']].round(3))


In [ ]:
# Visualisation des anomalies
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scatter rendement vs engrais
norm = df_result[df_result['anomalie'] == 0]
anom = df_result[df_result['anomalie'] == 1]

axes[0].scatter(norm['rendement_t_ha'], norm['engrais_kg_ha'],
                c='#2E8B57', alpha=0.6, s=60, label='Normal', edgecolors='white')
axes[0].scatter(anom['rendement_t_ha'], anom['engrais_kg_ha'],
                c='#dc2626', alpha=0.9, s=100, label='Anomalie', marker='X', zorder=4)
axes[0].set_xlabel('Rendement (t/ha)')
axes[0].set_ylabel('Engrais (kg/ha)')
axes[0].set_title('Détection d'anomalies — Isolation Forest')
axes[0].legend()

# Score d'anomalie
axes[1].hist(scores, bins=20, color='#2E8B57', alpha=0.7, edgecolor='white')
axes[1].axvline(iso.threshold_, color='red', linestyle='--',
                label=f'Seuil = {iso.threshold_:.3f}')
axes[1].set_xlabel('Score d'anomalie')
axes[1].set_ylabel('Fréquence')
axes[1].set_title('Distribution des scores d'anomalie')
axes[1].legend()

plt.tight_layout()
plt.savefig(os.path.join(MODELS_DIR, 'isolation_forest.png'), dpi=150, bbox_inches='tight')
plt.show()

# Sauvegarde
df_result.to_csv(os.path.join(MODELS_DIR, 'anomalies_detectees.csv'), index=False)
print("✓ Anomalies sauvegardées → anomalies_detectees.csv")


## 9. Synthèse et Recommandations

### Résultats des modèles


In [ ]:
print("╔══════════════════════════════════════════════════════╗")
print("║           SYNTHÈSE DES MODÈLES ML                   ║")
print("╠══════════════════════════════════════════════════════╣")
print(f"║  Random Forest  — R² = {r2_base:.3f}  MAE = {mae_base:.3f} t/ha         ║")
print(f"║  K-Means        — 3 segments, silhouette = {sil_scores[1]:.3f}       ║")
print(f"║  Prophet        — Prévisions 26 sem. (MAPE ~{mape*100:.1f}%)      ║")
print(f"║  Isolation Forest — {nb_ano} anomalies ({nb_ano/len(df_result)*100:.1f}% contamination)        ║")
print("╠══════════════════════════════════════════════════════╣")
print("║  RECOMMANDATIONS AGRONOMIQUES                        ║")
print("║  1. Priorité sol : indice_fertilite_sol est la       ║")
print("║     variable la plus corrélée au rendement           ║")
print("║  2. Surveillance hydrique : 25% des cycles en        ║")
print("║     déficit — systèmes d'irrigation recommandés      ║")
print("║  3. Parcelles sous-performance : plan d'amendement   ║")
print("║     du sol (pH + azote insuffisants)                 ║")
print("╚══════════════════════════════════════════════════════╝")


In [ ]:
# Figure de synthèse finale
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# RF : réel vs prédit
axes[0,0].scatter(y_test, y_pred_base, alpha=0.7, c='#2E8B57', s=60, edgecolors='white')
lims = [min(y_test.min(), y_pred_base.min()), max(y_test.max(), y_pred_base.max())]
axes[0,0].plot(lims, lims, 'r--', lw=2)
axes[0,0].set_title(f'Random Forest — R²={r2_base:.3f}')
axes[0,0].set_xlabel('Réel (t/ha)'); axes[0,0].set_ylabel('Prédit (t/ha)')

# K-Means : segments
for seg, label in labels_seg.items():
    g = agg[agg['segment']==seg]
    axes[0,1].scatter(g['rendement_moy'], g['indice_sol'],
                      c=couleurs_seg[seg], s=100, label=label, alpha=0.85, edgecolors='white')
axes[0,1].set_title('K-Means — Segmentation parcelles')
axes[0,1].set_xlabel('Rendement moy. (t/ha)'); axes[0,1].set_ylabel('Indice sol')
axes[0,1].legend(fontsize=8)

# Prophet : prix Maïs
axes[1,0].plot(df_mais['ds'], df_mais['y'], color='#2E8B57', lw=1.5, label='Historique')
axes[1,0].plot(forecast['ds'].tail(26), forecast['yhat'].tail(26), '--', color='#E76F51', lw=2, label='Prévision')
axes[1,0].fill_between(forecast['ds'].tail(26), forecast['yhat_lower'].tail(26),
                        forecast['yhat_upper'].tail(26), alpha=0.2, color='#E76F51')
axes[1,0].set_title('Prophet — Prix Maïs (26 sem.)'); axes[1,0].legend(fontsize=9)

# Anomalies
axes[1,1].scatter(norm['rendement_t_ha'], norm['eau_irrigation_m3_ha'],
                  c='#2E8B57', alpha=0.5, s=50, label='Normal', edgecolors='white')
axes[1,1].scatter(anom['rendement_t_ha'], anom['eau_irrigation_m3_ha'],
                  c='#dc2626', alpha=0.9, s=100, label='Anomalie', marker='X')
axes[1,1].set_title('Isolation Forest — Anomalies')
axes[1,1].set_xlabel('Rendement (t/ha)'); axes[1,1].set_ylabel('Eau irrigation (m³/ha)')
axes[1,1].legend(fontsize=9)

plt.suptitle('Synthèse des 4 Modèles ML — Ferme Agricole', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(MODELS_DIR, 'synthese_ml.png'), dpi=150, bbox_inches='tight')
plt.show()
print("✓ Figure de synthèse sauvegardée → synthese_ml.png")
